# OpenCodeReasoning Challenge

This notebook is the first of three notebooks that comprise my work for the OpenCodeReasoning challenge.

The inability of Large Language Models in the domains of Logic, Reasoning, Mathematics, and Coding has been a popular topic of discussion in the AI domain. The OpenCodeReasoning challenge addresses one aspect of this problem through the task of training a model to identify bugs in a code while given a statement of expectations from the code.

In this work, we fine-tune a 1.5B base "coder" model `Qwen/Qwen2.5-Coder-1.5B` using data from the OpenCodeReasoning dataset `nvidia/OpenCodeReasoning-2`. The model is tasked with providing commentary on a user's erroneous code and providing a correct solution while given a coding problem statement and accompanying erroneous code as inputs.

### First things first: the limitations
- Working in the free tier of Kaggle, we are limited in terms of GPU resources and runtime hours. This slows down and limits the amount of training we are able to perform. *This limitation in resources has limited the potential of this work.*
- For such a complex problem (coding), the LLM size is quite small (1.5B parameters). This could result in minimal progress, hallucinations, and unstable conversational behaviours.
- The dataset (as we will discuss) is not completely reliable and must be treated sceptically. It may be considered suitable for proof-of-concept purposes rather than definitive evaluation



# Training Plan:

The OpenCodeReasoning Challenge comes with 2 primary tasks expected from the model:
1. To identify problems in a user's buggy code.
2. To provide a correct solution for the problem statement.

To achieve this, I have chosen to use two different fine-tuning methods in succession to address both expectations. The sequence of steps taken in this challenge are as follows:
* **Dataset curation:**
    * Data from the `OpenCodeReasoning2` dataset is combined with external datasets to create a curated, more suitable dataset for this task
    * Different fields from these datasets are used as substitutes for User code, Human commentary, and Correct code to simulate a training scenario.
    * The curated dataset would also have sample input-output pairs, which would enable us to write assert statements to assess code correctness.
* **Supervised Fine-Tuning:**
    * To address the requirement (1) of identifying buggy code, I have chosen to use Supervised Fine-Tuning (SFT)
    * This requirement is a generative task, and SFT allows the model to learn to provide critique for the user's buggy code.
    * Simultaneously, it also allows the model to view various coding patterns and correct solutions for specific problems
* **Reinforcement Learning:**
    * To address requirement (2) of providing correct code solution, I have chosen to use Reinforcement Learning (RL) using Group-Relative Policy Optimisation (GRPO)
    * Reinforcement learning allows us to quantitatively score the correctness of code. This can in turn be used in a reward function, which then optimises the model to a desirable behaviour.

# This work

My attempt on the OpenCodeReasoning challenge spans 3 notebooks:
1. Data Curation (This notebook)
2. Stage 1: [SFT Fine-Tuning](https://www.kaggle.com/code/irishcoffeee/leap-la-santhoshm-sft-qwen-2-5-coder-1-5b)
3. Stage 2: [GRPO Fine-Tuning](https://www.kaggle.com/code/irishcoffeee/leap-la-grpo-finetuning-ocr2-santhoshm)

Both stages of fine-tuning use Unsloth for faster and more resource-efficient fine-tuning.

# Dataset Curation

In this notebook, we go through the steps I took to put the data together to use for the OpenCodeReasoning Challenge. Data is taken from [Nvidia's OpenCodeReasoning2 dataset](https://huggingface.co/datasets/nvidia/OpenCodeReasoning-2).

The dataset is too large, with about 2.5 million samples. Fortunately, this challenge only focuses on samples that use Python and have an "EASY" difficulty.

As the focus on this challenge is the ability of a model to debug buggy code, I chose to only include samples with buggy code. In context of the `nvidia/OpenCodeReasoning-2` dataset, this means only samples where `judgement='wrong'`. This concentrates the dataset, but also comes at a price - the model might be overfit to wrong code, so in the future it might not be able to tell if a code is actually correct.

To boil it down, we finally filter the dataset to:
* Language = Python
* Difficulty = EASY
* judgement = wrong


## How this dataset was created:
Two models were used in the creation of this dataset:
* Deepseek's R1
* Qwen's QWQ
There is a generator-evaluator setting

First, a large number of external datasets and accompanying data were pulled. R1 acted as the generator - it generates some thoughts and provides a solution. QWQ then looks at the question and R1's solution and provides some critique and judges whether the solution is correct or not.

### For this challenge
R1 will be treated as a human user and substitutes user-generated code data. QWQ's critique and judgement are treated as ground truth
**Important: This setting introduces heavy bias, as QWQ, just like any other LLM, is also prone to making mistakes.**



### Empty `question` column

In the `nvidia/OpenCodeReasoning-2` dataset (OCR2 dataset), all entries for the `question` column are empty. The dataset only contain proposed solution, critiques, etc. The creators of this dataset left the question column empty possibly to avoid copyright infringement. They provide (very messy and impractical) instructions to complete the dataset ourselves by querying other datasets.

For each sample, there is information on which external dataset it is from, which split in that dataset, and finally the index at which it can be found.
*Obviously, this is a very messy setup that is prone to obliterate the reliability of this dataset if one of the external datasets removes even one row from its bank*

So our first task here is to collect the questions (coding problem statements) from these external datasets and attach them to ours.

### Columns in the OCR2 dataset are as follows:

| Field | Explanation | Used for this challenge? |
| --- | --- | --- |
| id | unique id for each sample | no |
| question_id | unique id for each question | no, we use a combination of `dataset, split, index` to query instead |
| question | a dummy empty column | no |
| r1_generation | R1's response | no, we only use a portion of it. Found in the "solution" column |
| qwq_critique | QWQ's response | yes, this is treated as ground-truth |
| solution | code portion of R1's response | yes, treated as user's buggy code |
| judgement | judgement portion of QWQ's response | yes, to filter out samples with "wrong" code |
| dataset | name of the external dataset from which question is collected | yes |
| license | license associated with the dataset | no, although it should be considered as I am exposing my curated dataset. At present I do not understand the license completely, therefore this remains as a TODO |
| split | the 'split' of the external dataset | yes |
| difficulty | difficulty label for the question | yes |
| index | index to retrieve question from external datasets | yes |


#### About querying for Python samples
The dataset has samples of C++ and Python, separated by the 'split' rather than a column in the dataset. This makes querying for Python samples slightly different, by selecting by split name instead of field values.


# Getting started

Installing and loading useful libraries

In [2]:
%%capture
%pip install transformers==4.51.3 \
             datasets==3.5.0 \
             accelerate==1.6.0 \
             peft==0.15.2 \
             trl==0.16.1 \
             bitsandbytes==0.45.5 

In [3]:
from tqdm import tqdm
from datasets import load_dataset
from huggingface_hub import login

Using a huggingface token allows for faster dataset downloads and more.

In [4]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

login(token=hf_token)

For simplicity we will only take `train` splits from each dataset. There are only 746 wrong easy questions in `test` split and 15759 in `train` split

In [5]:
labels = ["nvidia/OpenCodeReasoning-2"]
data_dirs = [None]
splits = ["python"] # Select python split
idx = 0
label = labels[idx]
ds = load_dataset(label, data_dir = data_dirs[idx], cache_dir="./datasets",  split=splits[idx], streaming=True)

filtered = ds.filter(
    lambda x: x["judgement"] == 'wrong' and x["difficulty"]=='EASY' and
    x["split"] == 'train')

Resolving data files:   0%|          | 0/70 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/70 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [11]:
counts = {
}
indices = set()

for ocr2_ds_item in tqdm(filtered, desc="Counting number of easy questions with wrong code in each of the three datasets in train split"):
    if ocr2_ds_item["difficulty"].upper() == "EASY" and ocr2_ds_item["split"].lower() == "train" and ocr2_ds_item["judgement"] == "wrong":
        dataset_name = ocr2_ds_item["dataset"]
        external_dataset_index = ocr2_ds_item["index"]
        indices.add((dataset_name,external_dataset_index))
        counts[f"{dataset_name}"] = counts.get(f"{dataset_name}", 0) + 1


Counting number of easy questions with wrong code in each of the three datasets in train split: 15759it [08:31, 30.81it/s]


In [10]:
# Filtered what we needed, so deleting the bigger dataset to clear RAM
import gc
del ds
gc.collect()

703

In [12]:
counts

{'taco': 15759}

In [13]:
len(indices)

1527

In [14]:
unique_dataset_names = {s for s, _ in indices}
unique_dataset_names

{'taco'}

We can see that `counts` only has one key-value pair pointing to the TACO dataset.

`indices` indicates the unique problems found in the external datasets (here only TACO) for which at least one "wrong" solution exists in OCR2 dataset

And as a final confirmation by `unique_dataset_names`, all eligible samples are present in the TACO dataset.

Now storing relevant indices in a CSV file as a backup (This step is optional)

In [15]:
import csv

with open("wrong_soln_indices.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["dataset", "index"])  # header
    writer.writerows(indices)

# Create question bank

Now that we have the list of indices we need to retrieve from external datasets (which in our case is only TACO), we are ready to collect the dataset and select only the required samples and store the question statements in a bank. This way we may later query the bank to get the question.

In [21]:
taco_indices = [int(i) for _,i in indices]

In [19]:
# TACO train split:
ext_stream_train = load_dataset(
    "BAAI/TACO",
    split="train",
    streaming=False, # TACO is not compatible with streaming for some reason. It throws an error because it tries to open a URL as a local file
    trust_remote_code=True # TACO requires this to be true
)

In [22]:
ext_ds_train = ext_stream_train.select(taco_indices)


In [32]:
# loop over dataset and write each row immediately (STREAMING THE DATA TO A CSV)
with open("/kaggle/working/question_bank_taco_wrong_soln.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["index", "question"])
    for index, ext_ds_item in tqdm(zip(taco_indices, ext_ds_train), desc='Writing indices for "wrong" items from "taco" dataset "train" split to a csv', total=len(taco_indices)):
        question = ext_ds_item["question"]
        writer.writerow([index, question])


Writing indices for "wrong" items from "taco" dataset "train" split to a csv: 100%|██████████| 1527/1527 [00:00<00:00, 2068.25it/s]


# Create the dataset

Now we have all we need – the filtered OCR2 dataset, a bank of questions, and the TACO dataset. We may now start curating these into a new dataset with the required values.

## About the TACO dataset:
The BAAI/TACO dataset is considered a benchmark dataset for code generation models. Relevant fields in this dataset for our case are discussed below:

| Field | Explanation | Used for this challenge? |
| --- | --- | --- |
| question | the coding problem statement | YES |
| solutions | a list of valid code solutions | yes, this field is used to get a correct solution code for training purpose in stage-1 fine-tuning (SFT) |
| input_output | two lists of sample inputs and corresponding expected outputs | Yes, used for stage-2 of fine-tuning (GRPO) |


### Plan:
- We will iterate through the ocr2 dataset.
- We will query our question bank for the question.
- We will use the question above along with the `"solution"` column of the OCR2 dataset to act as the prompt. **[IMPORTANT: This `"solution"` is not the actual solution; it is a proposed solution (generated by R1) which in our case is used as the user's buggy code.]**
- For the response, we will go to the correct-solutions list in the TACO dataset and randomly select one correct solution and use it for training.
- We will then include sample inputs and sample outputs for evaluation purposes

So here I will generate a final data structure which includes:
```
      {
          'taco_question': taco_item['question'],
          'ocr_user_code': item['solution'],
          'ocr_qwq_critique': item['qwq_critique'],
          'taco_soln':random_soln,
          'inputs': inputs,
          'outputs': outputs,
          'judgement': item['judgement'],
          'debug_split': item['split'],
          'debug_index': item['index']
      }
```
where `taco_item` would be from the TACO dataset, and `item` would be from the OCR2 dataset

Each item in the above structure:
- `taco_question` - the coding problem statement
- `ocr_user_code` - acts as the user's buggy code
- `ocr_qwq_critique` - acts as critique for the user's buggy code
- `taco_soln` - a correct python code for the problem statement
- `inputs` - sample inputs to use for Reinforcement Learning in next finetuning stage
- `outputs` - sample outputs to use for Reinforcement Learning in next finetuning stage
- `judgement` - judgement by QWQ on whether the "user code" is right or wrong. Here it always remains "wrong" as we only picked erraneous code for our database.


Errors are handled by skipping broken samples. This is done by maintaining a set `skip`. If a particular combination of `split` and `index` points to a sample from which `question` could not be retrieved, we store the (`split`,`index`) pair here so that we may skip it the next time we encounter the same. This is due to the fact that each question in the TACO dataset is used multiple times in OCR2.

In [55]:
import traceback
import json
import random


taco_train_dict = dict(zip(taco_indices, ext_ds_train))


# max_samples = 4000
example_points = []
skip = set()
# iterator = iter(filtered)
try:
    # while len(example_points) < max_samples:
    for item in filtered:
        # item = next(iterator)
        if (item['split'],item['index']) in skip:
            continue

        taco_item = None
        taco_item = taco_train_dict[int(item['index'])]
            
        input_output = json.loads(taco_item['input_output'])
        inputs = input_output['inputs']
        outputs = input_output['outputs']

        try:
            solutions = json.loads(taco_item['solutions'])
        except ValueError:
            print("JSONDecodeError on split ",item['split']," index ",item['index'])
            skip.add((item['split'],item['index']))
            continue

        if solutions is None or solutions == []:
            print("Warning: No solutions for split ",item['split']," index ",item['index'])
            skip.add((item['split'], item['index']))
            continue
        random_soln = random.choice(solutions)

        example_points.append({
                'taco_question': taco_item['question'],
                'ocr_user_code': item['solution'],
                'ocr_qwq_critique': item['qwq_critique'],
                'taco_soln':random_soln,
                'inputs': inputs,
                'outputs': outputs,
                'judgement': item['judgement'],
                'debug_split': item['split'],
                'debug_index': item['index']
            })
except Exception as e:
    print("Error at split: ",item['split']," index:",item['index'], " skipping.")
    # print(item)
    traceback.print_exc()

## Upload to HuggingFace Hub
#### That's it! The dataset is now ready.

Time to upload it to HF Hub... But before that, we may consider splitting the dataset for `train` and `validation` sets

In [76]:
from datasets import Dataset
dataset = Dataset.from_list(example_points)
dataset.push_to_hub("santhosh-m/ocr2-wrong-solns-curated")

Uploading the dataset shards:   0%|          | 0/4 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/santhosh-m/ocr2-wrong-solns-curated/commit/f82edfa3fcfe4beceb0f55d759f9783f0c890ef2', commit_message='Upload dataset', commit_description='', oid='f82edfa3fcfe4beceb0f55d759f9783f0c890ef2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/santhosh-m/ocr2-wrong-solns-curated', endpoint='https://huggingface.co', repo_type='dataset', repo_id='santhosh-m/ocr2-wrong-solns-curated'), pr_revision=None, pr_num=None)

# Important consideration about the Train-Val split

As we can see above, there are only 1527 unique questions (`taco`) with multiple answers (`ocr2`) each. In such a case, it would be unwise to perform the Train-Val split on the `ocr2` dataset. Doing so will (have a 99% chance of) result in training the model with all available questions at least once. This means the model would have a ready solution for every question. Therefore, the Test-Val split MUST be done at the `taco` dataset level.

There is the option to use the `test` split of the TACO dataset as well. However, it is not guaranteed that those questions are different from the `train` split.

# Performing a 90-10 split

In [88]:
from datasets import Dataset


random.seed(42)  # reproducible
random.shuffle(taco_indices)

split_point = int(0.9 * len(taco_indices))

train_ids = set(taco_indices[:split_point])
val_ids   = set(taco_indices[split_point:])

train_data = [x for x in example_points if int(x["debug_index"]) in train_ids]
val_data   = [x for x in example_points if int(x["debug_index"]) in val_ids]

In [91]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
})

dataset.push_to_hub("santhosh-m/ocr2-wrong-solns-curated")

Uploading the dataset shards:   0%|          | 0/4 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


...

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/santhosh-m/ocr2-wrong-solns-curated/commit/8de2b1069e1f4d990a69a4bf2871390eb10f175f', commit_message='Upload dataset', commit_description='', oid='8de2b1069e1f4d990a69a4bf2871390eb10f175f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/santhosh-m/ocr2-wrong-solns-curated', endpoint='https://huggingface.co', repo_type='dataset', repo_id='santhosh-m/ocr2-wrong-solns-curated'), pr_revision=None, pr_num=None)

In [56]:
#### In case we need the dataset in json or csv form

# # export dataset as json
# json.dump(example_points, open("dataset.json", "w"), indent=4)

# # export dataset as csv
# with open("dataset.csv", "w", newline='', encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=example_points[0].keys(), quoting=csv.QUOTE_ALL)
#             # quoting=csv.QUOTE_ALL ensures multi-line fields are properly handled
#     writer.writeheader()
#     writer.writerows(example_points)

#### All done!
Now our data is ready to go for fine-tuning. Proceed to the next notebook [here](https://www.kaggle.com/code/irishcoffeee/leap-la-santhoshm-sft-qwen-2-5-coder-1-5b)